<a href="https://vigneashpandiyan.github.io/publications/Codes/" target="_blank" rel="noopener noreferrer">
  <img src="https://vigneashpandiyan.github.io/images/Link.png"
       style="max-width: 800px; width: 100%; height: auto;">
</a>

# Classification 1  - Training & Validating - MNIST

MNIST is a classic benchmark dataset of handwritten digits (0–9), widely used to learn and test image classification models.

Each image has 1 channel (i.e. monochrome images unlike RGB which has 3 channels).

Images are 28 × 28 pixel.

Each pixel is a value between 0–255.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np

Outline of this module:

1. Create a network
    - Arrange layers
    - Visualize layers
    - Creating loss function module
    - Creating optimizer module [Set learning rates here]
    
    
2. Data prepraration
    - Creating a data transformer
    - Downloading public dataset and applying transformation
    - Understanding dataset
    - Loading the transformed dataset [Set batch size and number of parallel processors here]
    
    
3. Setting up data - plotters


4. Training
    - Set Epoch
    - Train model
    
    
5. Validating
    - Overall-accuracy validation
    - Class-wise accuracy validation

    

## The Neural Network

In [ ]:
# 1.1 Creating a custom neural network
import torch.nn as nn
import torch.nn.functional as F

'''
Network arrangement

    Input -> Conv1 -> Relu -> Pool -> Conv2 -> Relu -> Pool -> FC1 -> Relu -> FC2 -> Relu -> FC3 -> Output

'''


class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5) # Reverted out_channels to 6 and kernel_size to 5
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.relu = nn.ReLU() # Activation function
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)
        self.fc1 = nn.Linear(16 * 4 * 4, 120)  # Corrected for 28x28 MNIST input
        self.fc2 = nn.Linear(120, 84) # In-channels, Out-Channels
        self.fc3 = nn.Linear(84, 10) # In-channels, Out-Channels

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(x)
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(-1, 16 * 4 * 4) # Corrected for 28x28 MNIST input
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x


net = Net()

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
net.to(device)
print(f"Model moved to {device}")

In [ ]:
# 1.2 Visualizing network
from torchsummary import summary
print("Network - ")
summary(net, (1, 28, 28)) # Updated input size for MNIST

In [ ]:
# 1.3. Creating loss function module
cross_entropy_loss = nn.CrossEntropyLoss()


In [ ]:
# 1.4. Creating optimizer module

optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
# 2.1. Creating data trasnformer

transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.1307,), (0.3081,))])

In [ ]:
# 2.2. Downloading public dataset and applying transformations simultaneously


# Download MNIST Training Dataset
trainset = torchvision.datasets.MNIST(root='./data', train=True,
                                        download=True, transform=transform)


# Download MNIST Testing Dataset
testset = torchvision.datasets.MNIST(root='./data', train=False,
                                       download=True, transform=transform)

In [ ]:
print("Train dataset - ", dir(trainset))
print("\n")
print("Test dataset - ", dir(testset))

## The Dataset

In [ ]:
# 2.3. - Understanding dataset

print("Number of training images - ", len(trainset.data))
print("Number of testing images - ", len(testset.data))
print("Size of images - ", trainset.data[0].shape)

In [ ]:
# 2.4. - Loading the transformed dataset

batch = 4
parallel_processors = 2

trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch,
                                          shuffle=True, num_workers=parallel_processors)


testloader = torch.utils.data.DataLoader(testset, batch_size=batch,
                                         shuffle=False, num_workers=parallel_processors)

# Class list
classes = ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')

In [ ]:
# 3. Setting up data plotters

# functions to show an image
def imshow(img):
    # Unnormalize for MNIST: img = img * std + mean
    mean = 0.1307
    std = 0.3081
    img = img * std + mean
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))


# get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# show images
imshow(torchvision.utils.make_grid(images))
plt.show()
print(labels)
print(' '.join('%5s' % classes[labels[j]] for j in range(4)))

## Plotting Training Loss and Accuracy

We add a new cell to plot the `train_losses` and `train_accuracies` over epochs using Matplotlib.


In [ ]:
from tqdm.notebook import tqdm

In [ ]:
# 4. Training
import time

num_epochs = 2

train_losses = []
train_accuracies = []

for epoch in range(num_epochs):  # loop over the dataset multiple times
    running_loss = 0.0
    correct_predictions_50 = 0
    total_samples_50 = 0
    epoch_start_time = time.time()

    pbar = tqdm(total=len(trainloader))
    for i, data in enumerate(trainloader):
        pbar.update();
        # Print data type in the first iteration of the first epoch
        if epoch == 0 and i == 0:
            print(f"Type of data in first iteration: {type(data)}")

        # get the inputs
        inputs, labels = data
        inputs = inputs.to(device)
        labels = labels.to(device)

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = net(inputs)
        loss = cross_entropy_loss(outputs, labels)
        loss.backward()
        optimizer.step()

        # update running loss
        running_loss += loss.item()

        # Calculate and store accuracy every 50 iterations
        _, predicted = torch.max(outputs.data, 1)
        total_samples_50 += labels.size(0)
        correct_predictions_50 += (predicted == labels).sum().item()

        if (i + 1) % 50 == 0: # Check every 50 mini-batches
            accuracy_50 = 100 * correct_predictions_50 / total_samples_50
            train_accuracies.append(accuracy_50)
            correct_predictions_50 = 0 # Reset counters
            total_samples_50 = 0

        if i % 2000 == 1999:    # print and store loss every 2000 mini-batches
            current_loss = running_loss / 2000
            train_losses.append(current_loss)
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, current_loss))
            running_loss = 0.0

    epoch_end_time = time.time()
    print(f"Epoch {epoch + 1} took {epoch_end_time - epoch_start_time:.2f} seconds")

print('Finished Training')
print(f"\nTrain Losses recorded: {train_losses}")
print(f"Train Accuracies recorded (every 50 iterations): {train_accuracies}")

# Plotting Loss and Accuracy Curves
plt.figure(figsize=(12, 5))

# Plotting Loss
plt.subplot(1, 2, 1) # 1 row, 2 columns, 1st plot
loss_iterations = [(j + 1) * 2000 for j in range(len(train_losses))]
plt.plot(loss_iterations, train_losses, label='Training Loss')
plt.xlabel('Iteration (x2000)')
plt.ylabel('Loss')
plt.title('Training Loss over Iterations')
plt.legend()
plt.grid(False)

# Plotting Accuracy
plt.subplot(1, 2, 2) # 1 row, 2 columns, 2nd plot
accuracy_iterations = [(j + 1) * 50 for j in range(len(train_accuracies))]
plt.plot(accuracy_iterations, train_accuracies, label='Training Accuracy', color='orange')
plt.xlabel('Iteration (x50)')
plt.ylabel('Accuracy (%)')
plt.title('Training Accuracy over Iterations')
plt.legend()
plt.grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# 5.1 Overall-accuracy Validation
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 10000 test images: %d %%' % (
    100 * correct / total))

In [ ]:
# 5.2 Classwise-accuracy Validation
class_correct = list(0. for i in range(10))
class_total = list(0. for i in range(10))
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images = images.to(device)
        labels = labels.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        c = (predicted == labels).squeeze()
        for i in range(4):
            label = labels[i]
            class_correct[label] += c[i].item()
            class_total[label] += 1


for i in range(10):
    print('Accuracy of %5s : %2d %%' % (
        classes[i], 100 * class_correct[i] / class_total[i]))

##Plotting Confusion Matrix
We add a new cell to calculate the confusion matrix using the test data and plot it using Matplotlib and Seaborn to visualize class-wise performance.


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Get all predictions and true labels from the test set
all_labels = []
all_predicted = []
with torch.no_grad():
    for data in testloader:
        images, labels = data
        images = images.to(device)
        outputs = net(images)
        _, predicted = torch.max(outputs, 1)
        all_labels.extend(labels.cpu().numpy())
        all_predicted.extend(predicted.cpu().numpy())

# Compute the confusion matrix
cm = confusion_matrix(all_labels, all_predicted)

# Plot the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()